In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CarNoncar.zip to CarNoncar.zip


In [ ]:
import zipfile
import os
import cv2
import h5py
import numpy as np

In [ ]:
with zipfile.ZipFile("CarNoncar.zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Dataset extracted")

Dataset extracted


In [ ]:
print(os.listdir("CarNoncar"))

['NonCar', 'Car']


In [ ]:
IMG_SIZE = 128

images = []
labels = []

CAR_FOLDER = "CarNoncar/Car"
NON_CAR_FOLDER = "CarNoncar/NonCar"

for file in os.listdir(CAR_FOLDER):

    path = os.path.join(CAR_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(1)


for file in os.listdir(NON_CAR_FOLDER):

    path = os.path.join(NON_CAR_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(0)

X = np.array(images, dtype=np.float32)
Y = np.array(labels)

X = X / 255.0

print("X shape:", X.shape)
print("Y shape:", Y.shape)

with h5py.File("data.h5", "w") as hf:

    hf.create_dataset("X", data=X)
    hf.create_dataset("Y", data=Y)

print("data.h5 created successfully")

X shape: (200, 128, 128, 3)
Y shape: (200,)
data.h5 created successfully


In [ ]:
with h5py.File("data.h5", "r") as hf:

    X = hf["X"][:]
    Y = hf["Y"][:]

print(X.shape)
print(Y.shape)

(200, 128, 128, 3)
(200,)


In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
from sklearn.utils import shuffle
X, Y = shuffle(X, Y, random_state=42)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.5375 - loss: 0.7928 - val_accuracy: 0.4500 - val_loss: 0.6970
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.7063 - loss: 0.6407 - val_accuracy: 0.6500 - val_loss: 0.5960
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8625 - loss: 0.4888 - val_accuracy: 0.6750 - val_loss: 0.5896
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.7875 - loss: 0.4194 - val_accuracy: 0.7250 - val_loss: 0.5889
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8625 - loss: 0.3356 - val_accuracy: 0.7250 - val_loss: 0.5554
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.8562 - loss: 0.3298 - val_accuracy: 0.7750 - val_loss: 0.5313
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9125 - loss: 0.2201 - val_accuracy: 0.7750 - val_loss: 0.5527
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9375 - loss: 0.1727 - val_accuracy: 0.7500 - val_loss: 0.5440
Epoch 9/10
5/5 

In [ ]:
model.evaluate(x_test, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.8000 - loss: 0.8009


[0.8009496927261353, 0.800000011920929]

In [ ]:
model.save("car_classifier.h5")
print("Model saved")

Model saved
